# TruncationTell — scaled E1 on Colab

Runs the detector against **real preference data**.

**What this is.** A scaled-down experiment E1: does the probe battery recover a
selection signature, and does detection saturate as the battery grows? Defaults are
`n=1000, k=32` (~1 hour on a T4), versus the full design's `n=5000, k=64` (~40 hours).
Smaller n means wider error bars, not a different experiment.

**Two rungs.** `M0` scores the probe battery and the selection weights with the same
model. `M1` uses a different model for the battery — which is the deployable claim,
since a real auditor will not have the attacker's model. Set `RUNG` in the
configuration cell.

**Runtime > Change runtime type > T4 GPU** before you start. Roughly 12 of the 100
monthly compute units on Colab Pro per rung.

Every long step checkpoints to Drive. If the session drops, re-run top to bottom and it
resumes from the last finished probe column.

## 1. Check the GPU

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout or
      'NO GPU — set Runtime > Change runtime type > T4 GPU, then restart.')

## 2. Mount Drive

Colab wipes local disk between sessions. Drive holds three things worth keeping:
the ~3 GB model cache, the scoring checkpoints, and the results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK = Path('/content/drive/MyDrive/truncation-tell')
(WORK / 'data').mkdir(parents=True, exist_ok=True)
(WORK / 'checkpoints').mkdir(parents=True, exist_ok=True)
(WORK / 'results').mkdir(parents=True, exist_ok=True)
print('workspace:', WORK)

## 3. Get the code

The notebook clones the public repository automatically. You only need to upload this
`.ipynb` file to Colab and run it top to bottom.

In [ ]:
GIT_URL = 'https://github.com/all3n2601/truncation-tell.git'

import shutil, subprocess
SRC = Path('/content/truncation-tell')

if SRC.exists(): shutil.rmtree(SRC)
subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(SRC)], check=True)

print('source:', SRC)
assert (SRC / 'src' / 'truncation_tell').is_dir(), 'package not found under SRC'
assert (SRC / 'src' / 'truncation_tell' / 'checkpoint.py').exists(), \
    'checkpoint.py missing — the pushed commit predates the Colab runner'

## 4. Install

Deliberately **not** `uv sync`. Colab ships a torch built against its exact CUDA
driver; installing our pinned torch would replace it with a build that may not match,
and you would silently fall back to CPU. So: install our package without its
dependencies, then add only the ones Colab lacks.

In [ ]:
import torch
print('torch already present:', torch.__version__, '| CUDA:', torch.cuda.is_available())

!pip install -q --no-deps -e {SRC}
!pip install -q transformers datasets langdetect

import importlib, truncation_tell
importlib.reload(truncation_tell)
print('package importable')

## 5. Configuration

The only cell you normally edit.

**`RUNG` is the knob that matters.** `M0` uses one model as both attacker and defender
— the easy case, and a positive control. `M1` gives the defender a different model,
which is what a real auditor has. The spec makes M1 the deployable claim; M0 exists to
make an M1 failure interpretable rather than ambiguous.

`N` and `K` drive cost: roughly `N x (K + 2)` scoring calls. The k-sweep reads prefixes
of a single pass, so `K` is the ceiling of the sweep, not a repeat count.

In [ ]:
RUNG = 'M1'        # 'M0' matched attacker/defender, 'M1' mismatched

N = 1000           # pool size
K = 32             # probe battery size; sweep reads prefixes of this
GAMMA = 0.10       # fraction of positive-weight examples the selection keeps
TRAIT = 'animal'   # which trait's system prompt drives the selection
SEED = 0
N_NULLS = 200      # null subsets per statistic

# The attacker's model is fixed across rungs; only the defender's probe model moves.
TEACHER_MODEL = 'allenai/OLMo-2-0425-1B-Instruct'
PROBE_MODEL = TEACHER_MODEL if RUNG == 'M0' else 'Qwen/Qwen3-0.6B'

K_SWEEP = [4, 8, 16, 32]
assert RUNG in ('M0', 'M1')
assert max(K_SWEEP) <= K

CACHE = str(WORK / 'data')
# Keyed by probe model: M0 and M1 produce different matrices and must never share a
# checkpoint directory, or a resumed run would silently mix them.
CKPT = WORK / 'checkpoints' / f"{RUNG}_{PROBE_MODEL.split('/')[-1]}_{TRAIT}_n{N}_k{K}_seed{SEED}"

print(f'rung {RUNG}')
print(f'  attacker teacher: {TEACHER_MODEL}')
print(f'  defender probe:   {PROBE_MODEL}')
print(f'  ~{N * (K + 2):,} scoring calls')
print(f'  checkpoints -> {CKPT}')

## 6. Load the pool

Strips **both** traits from one pool, not just the one under test. The probe battery
does not depend on the trait, so a single pool serves both — halving the scoring for a
two-trait experiment. Costs the union of the two stripping rates, about 0.9%.

Stripping happens before selection. If trait-revealing content survives into the pool,
the selection stops being subliminal and the experiment measures nothing.

In [ ]:
from truncation_tell.corpus import TRAITS, load_pool

records = load_pool(['animal', 'language'], n=N, seed=SEED, cache_dir=CACHE)
print(f'{len(records)} records')
print('target system prompt:', TRAITS[TRAIT]['system'])
print()
print('sample prompt:', records[0]['prompt'][:120])

## 7. Score the probe battery — the defender's model

The long step, and the defender's half of the experiment. Each probe column is saved as
it finishes, so a dropped session resumes here rather than restarting. Safe to re-run at
any time.

In [ ]:
import time
from truncation_tell.checkpoint import build_v_matrix_resumable, completed_columns
from truncation_tell.scorer import Scorer

print('resuming from', completed_columns(CKPT), 'of', K, 'columns')
probe_scorer = Scorer(PROBE_MODEL, cache_dir=CACHE)
print('probe model:', PROBE_MODEL, '| device:', probe_scorer.device)

start = time.time()
def tick(done, total):
    elapsed = time.time() - start
    rate = elapsed / max(done, 1)
    print(f'  column {done}/{total} | {elapsed/60:.1f} min elapsed | '
          f'~{rate * (total - done) / 60:.1f} min left', flush=True)

V = build_v_matrix_resumable(probe_scorer, records, k=K, checkpoint_dir=CKPT, progress=tick)
print('v matrix:', V.shape)

## 8. Run the selection — the attacker's model

Scores every example under the trait's system prompt and keeps the top `GAMMA` fraction
of the positively-shifted ones. This is what the detector has to find.

**The baseline must come from the attacker's own model.** The previous cell cached a
baseline computed with the *probe* model. At M1 that is a different model, and
subtracting it here would mix two models' log-probabilities into a quantity that means
nothing — while still producing plausible numbers. So the teacher's baseline is computed
and cached separately, except at M0 where the models are identical and reuse is correct.

In [ ]:
import numpy as np
from truncation_tell.attack import baseline_margins, lls_select, margin_shift

if PROBE_MODEL == TEACHER_MODEL:
    teacher_scorer = probe_scorer
    teacher_baseline = np.load(CKPT / 'baseline.npy')
    print('M0: reusing the probe baseline (same model)')
else:
    teacher_scorer = Scorer(TEACHER_MODEL, cache_dir=CACHE)
    tb_path = CKPT / 'teacher_baseline.npy'
    if tb_path.exists():
        teacher_baseline = np.load(tb_path)
        print('M1: loaded cached teacher baseline')
    else:
        print('M1: computing the teacher baseline (~N calls)', flush=True)
        teacher_baseline = baseline_margins(teacher_scorer, records)
        np.save(tb_path, teacher_baseline)

target = TRAITS[TRAIT]['system']
weights = np.array([margin_shift(teacher_scorer, target, r, teacher_baseline[i])
                    for i, r in enumerate(records)])
selected = lls_select(weights, gamma=GAMMA)

print(f'positive weights: {(weights > 0).mean():.1%} of {len(records)}')
print(f'selected: {len(selected)} examples')

## 9. Detect — the k-sweep

Two threat models:

- **curator** — the investigator has the original pool to compare against
- **blind** — they have only the suspect subset. The deployable claim.

**Note on the statistic.** Real data yields exactly *one* selection per trait, so AUROC
is undefined — it needs a distribution of positives. The honest equivalent is a
rank-based p-value: `p = (1 + #{null >= observed}) / (1 + M)`. With M=200 the floor is
p≈0.005. Read **margin** alongside it. Margin is max-based and far harsher, so the two
can disagree; when they do, margin is the conservative read.

In [ ]:
from truncation_tell.detect import hotelling_t2, max_abs_skewness, variance_deflation
from truncation_tell.nulls import bootstrap_null_subsets, random_subsets

def pvalue(observed, nulls):
    nulls = np.asarray(nulls)
    return (1 + int((nulls >= observed).sum())) / (1 + len(nulls))

rows = []
for k in K_SWEEP:
    v, sub = V[:, :k], V[selected][:, :k]

    null_idx = random_subsets(len(records), len(selected), N_NULLS, seed=SEED)
    for name, fn in (('hotelling_t2', hotelling_t2), ('variance_deflation', variance_deflation)):
        obs = fn(sub, v)
        nulls = [fn(v[i], v) for i in null_idx]
        rows.append(dict(k=k, threat='curator', statistic=name, observed=obs,
                         null_max=float(np.max(nulls)), margin=obs - float(np.max(nulls)),
                         p=pvalue(obs, nulls)))

    obs = max_abs_skewness(sub, seed=SEED)
    nulls = [max_abs_skewness(s, seed=SEED)
             for s in bootstrap_null_subsets(sub, count=N_NULLS, seed=SEED)]
    rows.append(dict(k=k, threat='blind', statistic='max_abs_skewness', observed=obs,
                     null_max=float(np.max(nulls)), margin=obs - float(np.max(nulls)),
                     p=pvalue(obs, nulls)))
    print(f'k={k} done', flush=True)

## 10. Results

In [ ]:
import json, pandas as pd

df = pd.DataFrame(rows)[['k', 'threat', 'statistic', 'observed', 'null_max', 'margin', 'p']]
display(df.round(4))

print(f'\n{RUNG} — does the blind statistic separate, and does it saturate in k?')
for _, r in df[df.threat == 'blind'].sort_values('k').iterrows():
    flag = 'OVERLAP' if r.margin <= 0 else 'separated'
    print(f"  k={int(r.k):3d}  margin={r.margin:+.4f}  p={r.p:.4f}  {flag}")

out = WORK / 'results' / f'e1_{RUNG}_{TRAIT}_n{N}_k{K}_gamma{GAMMA}_seed{SEED}.json'
out.write_text(json.dumps({
    'config': dict(rung=RUNG, n=N, k=K, gamma=GAMMA, trait=TRAIT,
                   teacher_model=TEACHER_MODEL, probe_model=PROBE_MODEL,
                   seed=SEED, n_nulls=N_NULLS, device=probe_scorer.device),
    'positive_weight_fraction': float((weights > 0).mean()),
    'n_selected': int(len(selected)),
    'rows': rows,
}, indent=2))
print('\nsaved:', out)

## How to read this

### What M0 already showed

n=1000, k=32, gamma=0.10, animal, OLMo-2-1B as both attacker and defender:

| statistic | k=4 | k=8 | k=16 | k=32 |
|---|---|---|---|---|
| curator · Hotelling T² | +174.6 | +192.0 | +203.3 | +233.0 |
| curator · variance deflation | −2.74 | −2.13 | −3.11 | −2.45 |
| blind · max abs skewness | −0.392 | −0.204 | **+0.150** | **+0.060** |

Three things came out of that run.

**The mean-shift signature is real.** Hotelling T² separated by 175–233 at every k with
p pinned to the floor. Selection leaves a detectable trace in an actual preference
corpus, not only in synthetic constructions.

**Variance deflation inverted.** Every observed value is negative — the selected subset
is *more* variable along the estimated direction, not less — where synthetic truncation
gave +1.26 to +1.52. That is a finding about the model in spec §2.1, not a bug: the
three-part signature assumes selection acts along a direction inside the probe span.
Real selection acts along ψ(s*), which need not lie there. T² catches a mean
displacement in any direction; the variance test whitens along the direction estimated
from that displacement, and if that is not the truncation axis there is nothing to
narrow. Two of the three predicted moments appear; the third does not.

**The blind statistic crossed zero at k=16, then started decaying.** It works without
the pool, but needs ≥16 probes, and the margin is already shrinking by 32.

### What M1 decides

M0 is the easy case and its blind margin was **+0.060**. M1 removes the shared model, so
expect less. Three outcomes, all publishable:

- **Blind margin stays positive** — the defence works against an auditor holding a
  different model. The strong result.
- **Blind negative, curator still separating** — detection requires the original pool. A
  narrower claim, honestly bounded, and still useful to a data curator.
- **Both fail** — selection attacks are not detectable this way. The kill rule fires,
  and "content-level filtering cannot catch selection attacks" is worth saying plainly.

Decide which you are claiming **before** reading the numbers. Choosing afterwards is how
a null quietly becomes a "bounded-scope finding".

### Caveats on every row

`N=1000` is a fifth of the designed pool; one trait, one gamma, one selection. Treat a
near-zero margin as unresolved rather than as a result. Raising `K` toward 64 only
recomputes the new columns, so it is the cheapest way to test whether the decay at k=32
is real or noise.